1. Combine all features (rdkit,MORDRED,padel,TDA)
2. Clean all labels: combine all same core meaning labels.
3. generate the full dataset: combine features and label togehter


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
'''
Plotting libraries
'''
import pandas as pd
import matplotlib.cm as cm
#from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.cluster import KMeans
from sklearn import datasets, decomposition
from sklearn.manifold import TSNE

from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_classification

https://github.com/ecrl/padelpy

https://onlinelibrary.wiley.com/doi/10.1002/jcc.21707

In [3]:
!pip install iterative-stratification
!pip install nltk
!pip install fuzzywuzzy

In [4]:
!cp /content/drive/MyDrive/DOS_0.25n/Helvetica-Font/Helvetica.ttf /usr/local/lib/python3.10/dist-packages/matplotlib/mpl-data/fonts/ttf
from matplotlib.font_manager import FontProperties
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import matplotlib.pyplot as plt
myfont = FontProperties(fname=r'/usr/local/lib/python3.10/dist-packages/matplotlib/mpl-data/fonts/ttf/Helvetica.ttf',size=20)

# **1.Load dataset**

### Common molecules

In [5]:
com = pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/all6.csv')
com_f=com.loc[:, (com!= 0).any(axis=0)]   # remove all zero value  label
com_f

,name,IsomericSMILES,CID,'acid','alcohol','alcoholic','aldehidic','aldehydic','alliaceous (onion,'alliaceous',...,'walnut','warm','waxy','whiskey','wine-like','winelike','winey','wood','woody',garlic)'
0,acetaldehyde,CC=O,177.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,13002-09-0,CC(C)CCOC(C)OCCC(C)C,83036.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,acetal,CCOC(C)OCC,7765.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,63449-64-9,CC/C=C\CCOC(OCC/C=C\CC)C,5367767.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"1,1-dipropoxyethane",CCCOC(C)OCCC,66929.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,l-valine,CC(C)[C@@H](C(=O)O)N,6287.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1507,diethyl phthalate,CCOC(=O)C1=CC=CC=C1C(=O)OCC,6781.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1508,phenethyl anthranilate,C1=CC=C(C=C1)CCOC(=O)C2=CC=CC=C2N,8615.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1509,pyrrolidine,C1CCNC1,31268.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# **2. Features combination**

a.	add TDA in common mol features

b.	add TDA to unique mole features

c.	combine as full dataset

d. data analysis


e.	feature selection


# Data load

###  Combine for common MOL data's features

In [9]:
pad=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/com_pad.csv')
pad_f=pad.drop(pad.columns[[0, 1, 2]],axis=1)
mor=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/com_mored.csv')
mor_f=mor.drop(mor.columns[[0]],axis=1)
rd=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/com_rdkit.csv')
rd_f=rd.drop(rd.columns[[0]],axis=1)
TDA=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/TDA.csv')
TDA

<ipython-input-9-77e244adb3f5>:3: DtypeWarning: Columns (139,148,157,166,175,184,193,202,211,220,229,346,355,364,373,382,391,400,409,418,427,436,445,453,461,466,467,468,469,477,485,493,501,509,517,525,533,541,549,557,562,563,564,565,573,581,589,597,605,613,621,629,637,869,885,1350) have mixed types. Specify dtype option on import or set low_memory=False.
  mor=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/com_mored.csv')


,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,1.777983,0.000000,0.000000
1,3.606813,1.193387,0.000000
2,2.999655,0.000000,0.000000
3,3.681115,1.027111,0.000000
4,3.236267,0.000000,0.000000
...,...,...,...
1506,2.853617,1.355357,0.000000
1507,3.345647,1.552718,0.000000
1508,3.435885,1.277218,0.000000
1509,2.553369,0.244144,0.000000


In [10]:
f_all=pd.concat([rd_f,pad_f,mor_f,TDA],axis=1)
f_all

,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,NumRadicalElectrons,...,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,name,IsomericSMILES,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,9.569444,-2.722222,9.569444,1.506944,0.355008,44.053,40.021,44.026215,18,0,...,0,6.0,4.0,2.250000,1.000000,CC=O,177.0,1.777983,0.000000,0.000000
1,7.970706,-4.726701,7.970706,4.177266,0.561161,202.338,176.130,202.193280,86,0,...,12,56.0,55.0,6.833333,3.333333,CC(C)CCOC(C)OCCC(C)C,83036.0,3.606813,1.193387,0.000000
2,7.456250,-3.900000,7.456250,3.527176,0.519424,118.176,104.064,118.099380,50,0,...,6,28.0,27.0,4.111111,2.166667,CCOC(C)OCC,7765.0,2.999655,0.000000,0.000000
3,7.987270,-4.530029,7.987270,1.951380,0.317337,226.360,200.152,226.193280,94,0,...,14,60.0,59.0,6.111111,4.166667,CC/C=C\CCOC(OCC/C=C\CC)C,5367767.0,3.681115,1.027111,0.000000
4,7.664583,-4.225000,7.664583,3.723747,0.534645,146.230,128.086,146.130680,62,0,...,8,36.0,35.0,4.611111,2.666667,CCCOC(C)OCCC,66929.0,3.236267,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48,0,...,8,32.0,33.0,5.333333,1.888889,CC(C)[C@@H](C(=O)O)N,6287.0,2.853617,1.355357,0.000000
1507,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86,0,...,22,72.0,81.0,6.444444,3.916667,CCOC(=O)C1=CC=CC=C1C(=O)OCC,6781.0,3.345647,1.552718,0.000000
1508,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92,0,...,23,86.0,96.0,5.444444,4.138889,C1=CC=C(C=C1)CCOC(=O)C2=CC=CC=C2N,8615.0,3.435885,1.277218,0.000000
1509,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30,0,...,0,20.0,20.0,1.250000,1.250000,C1CCNC1,31268.0,2.553369,0.244144,0.000000


In [11]:
f_smile=f_all.drop(columns=['IsomericSMILES','name'])
f_smile

,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,NumRadicalElectrons,...,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,9.569444,-2.722222,9.569444,1.506944,0.355008,44.053,40.021,44.026215,18,0,...,6.289459,4,0,6.0,4.0,2.250000,1.000000,1.777983,0.000000,0.000000
1,7.970706,-4.726701,7.970706,4.177266,0.561161,202.338,176.130,202.193280,86,0,...,5.054832,397,12,56.0,55.0,6.833333,3.333333,3.606813,1.193387,0.000000
2,7.456250,-3.900000,7.456250,3.527176,0.519424,118.176,104.064,118.099380,50,0,...,5.368154,75,6,28.0,27.0,4.111111,2.166667,2.999655,0.000000,0.000000
3,7.987270,-4.530029,7.987270,1.951380,0.317337,226.360,200.152,226.193280,94,0,...,5.385554,631,14,60.0,59.0,6.111111,4.166667,3.681115,1.027111,0.000000
4,7.664583,-4.225000,7.664583,3.723747,0.534645,146.230,128.086,146.130680,62,0,...,5.218953,149,8,36.0,35.0,4.611111,2.666667,3.236267,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48,0,...,6.162052,65,8,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
1507,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86,0,...,7.402974,446,22,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
1508,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92,0,...,7.306372,702,23,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
1509,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30,0,...,5.076679,15,0,20.0,20.0,1.250000,1.250000,2.553369,0.244144,0.000000


In [ ]:
f_smile.to_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/feature_c.csv',index=False)

In [12]:
## download label data
label= pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/all6.csv')
#label=label.drop(label.columns[[0, 1, 2]],axis=1)
label

,name,IsomericSMILES,CID,'acid','alcohol','alcoholic','aldehidic','aldehydic','alliaceous (onion,'alliaceous',...,'walnut','warm','waxy','whiskey','wine-like','winelike','winey','wood','woody',garlic)'
0,acetaldehyde,CC=O,177.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,13002-09-0,CC(C)CCOC(C)OCCC(C)C,83036.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,acetal,CCOC(C)OCC,7765.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,63449-64-9,CC/C=C\CCOC(OCC/C=C\CC)C,5367767.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"1,1-dipropoxyethane",CCCOC(C)OCCC,66929.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,l-valine,CC(C)[C@@H](C(=O)O)N,6287.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1507,diethyl phthalate,CCOC(=O)C1=CC=CC=C1C(=O)OCC,6781.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1508,phenethyl anthranilate,C1=CC=C(C=C1)CCOC(=O)C2=CC=CC=C2N,8615.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1509,pyrrolidine,C1CCNC1,31268.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
com[['name','IsomericSMILES','CID']]

,name,IsomericSMILES,CID
0,acetaldehyde,CC=O,177.0
1,13002-09-0,CC(C)CCOC(C)OCCC(C)C,83036.0
2,acetal,CCOC(C)OCC,7765.0
3,63449-64-9,CC/C=C\CCOC(OCC/C=C\CC)C,5367767.0
4,"1,1-dipropoxyethane",CCCOC(C)OCCC,66929.0
...,...,...,...
1506,l-valine,CC(C)[C@@H](C(=O)O)N,6287.0
1507,diethyl phthalate,CCOC(=O)C1=CC=CC=C1C(=O)OCC,6781.0
1508,phenethyl anthranilate,C1=CC=C(C=C1)CCOC(=O)C2=CC=CC=C2N,8615.0
1509,pyrrolidine,C1CCNC1,31268.0


In [14]:
f_com_all=pd.concat([com[['name','IsomericSMILES','CID']],f_smile],axis=1)
f_com_all

,name,IsomericSMILES,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,...,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,acetaldehyde,CC=O,177.0,9.569444,-2.722222,9.569444,1.506944,0.355008,44.053,40.021,...,6.289459,4,0,6.0,4.0,2.250000,1.000000,1.777983,0.000000,0.000000
1,13002-09-0,CC(C)CCOC(C)OCCC(C)C,83036.0,7.970706,-4.726701,7.970706,4.177266,0.561161,202.338,176.130,...,5.054832,397,12,56.0,55.0,6.833333,3.333333,3.606813,1.193387,0.000000
2,acetal,CCOC(C)OCC,7765.0,7.456250,-3.900000,7.456250,3.527176,0.519424,118.176,104.064,...,5.368154,75,6,28.0,27.0,4.111111,2.166667,2.999655,0.000000,0.000000
3,63449-64-9,CC/C=C\CCOC(OCC/C=C\CC)C,5367767.0,7.987270,-4.530029,7.987270,1.951380,0.317337,226.360,200.152,...,5.385554,631,14,60.0,59.0,6.111111,4.166667,3.681115,1.027111,0.000000
4,"1,1-dipropoxyethane",CCCOC(C)OCCC,66929.0,7.664583,-4.225000,7.664583,3.723747,0.534645,146.230,128.086,...,5.218953,149,8,36.0,35.0,4.611111,2.666667,3.236267,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,l-valine,CC(C)[C@@H](C(=O)O)N,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,...,6.162052,65,8,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
1507,diethyl phthalate,CCOC(=O)C1=CC=CC=C1C(=O)OCC,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,...,7.402974,446,22,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
1508,phenethyl anthranilate,C1=CC=C(C=C1)CCOC(=O)C2=CC=CC=C2N,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,...,7.306372,702,23,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
1509,pyrrolidine,C1CCNC1,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,...,5.076679,15,0,20.0,20.0,1.250000,1.250000,2.553369,0.244144,0.000000


In [15]:
f_com_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Columns: 3915 entries, name to PERSISTENCE_ENTROPY_DIM_2
dtypes: bool(2), float64(2793), int64(459), object(661)
memory usage: 45.1+ MB


In [16]:
duplicate_cols = f_com_all.columns[f_com_all.columns.duplicated()]
duplicate_cols

Index(['TPSA', 'nAcid', 'nBase', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nH',
       'nB', 'nC', 'nN',
       ...
       'JGI6', 'JGI7', 'JGI8', 'JGI9', 'JGI10', 'VAdjMat', 'MWC10', 'SRW10',
       'MW', 'AMW'],
      dtype='object', length=694)

In [ ]:
val=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/val_rdkit.csv')
val

,Unnamed: 0,SMILES,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,CCC(C)C(=O)OC1CC2CCC1(C)C2(C)C,22082179.0,13.497842,-4.996768,13.497842,3.089241,0.699193,238.371,212.163,...,0,0,0,0,0,0,0,0,0,0
1,1,CC(C)C1CCC(C)CC1OC(=O)CC(C)O,11586787.0,13.164320,-5.265960,13.164320,3.077813,0.771037,242.359,216.151,...,0,0,0,0,0,0,0,0,0,0
2,2,CC(=O)/C=C/C1=CCC[C@H](C1(C)C)C,6433147.0,12.093950,-4.469749,12.093950,1.899280,0.611768,192.302,172.142,...,0,0,0,0,0,0,0,0,0,0
3,3,CC(=O)OCC(COC(=O)C)OC(=O)C,5541.0,11.597676,-4.409421,11.597676,2.359254,0.478698,218.205,204.093,...,0,0,0,0,0,0,0,0,0,0
4,4,CCCCCCCC(=O)OC/C=C(/CCC=C(C)C)\C,6365696.0,12.947147,-4.827799,12.947147,2.013318,0.280285,280.452,248.196,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732,732,c1cc(sc1)SSc1cccs1,23347.0,7.634320,-0.080798,7.634320,0.030352,0.706263,230.404,224.356,...,0,0,0,0,0,0,0,2,0,0
733,733,C[C@@H]1CCC[C@]2(C)CCCC[C@@]12O,15559490.0,8.688542,-4.860417,8.688542,4.173333,0.610008,182.307,160.131,...,0,0,0,0,0,0,0,0,0,0
734,734,CCC(C(=O)OCCC(C)C)C,520326.0,12.484746,-4.461305,12.484746,2.786173,0.595778,172.268,152.108,...,0,0,0,0,0,0,0,0,0,0
735,735,CC(=CCCC(C)CC=O)C,7794.0,11.348354,-4.384630,11.348354,1.856918,0.439269,154.253,136.109,...,0,0,0,0,0,0,0,0,0,0


# Combine unique MOL with commom dataset

In [18]:
or_f=pd.read_csv('/content/drive/MyDrive/pgml/f_data_1701.csv') #0611_combine_features.ipynb
or_f

<ipython-input-18-2e433d010988>:1: DtypeWarning: Columns (2092,2093,2094,2095,2096,2097,2098,2099,2100,2101,2102,2103,2163,2164,2165,2166,2167,2172,2173,2174,2175,2176,2181,2182,2183,2184,2185,2190,2191,2192,2193,2194,2199,2200,2201,2202,2203,2208,2209,2210,2211,2212,2271,2272,2273,2274,2275,2280,2281,2282,2283,2284,2289,2290,2291,2292,2293,2298,2299,2300,2301,2302,2307,2308,2309,2310,2311,2316,2317,2318,2319,2320,2324,2325,2326,2327,2328,2332,2333,2334,2335,2336,2340,2341,2342,2343,2344,2348,2349,2350,2351,2352,2356,2357,2358,2359,2360,2364,2365,2366,2367,2368,2372,2373,2374,2375,2376,2380,2381,2382,2383,2384,2388,2389,2390,2391,2392,2396,2397,2398,2399,2400,2404,2405,2406,2407,2408,2412,2413,2414,2415,2416,2420,2421,2422,2423,2424,2425,2426,2427,2428,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2440,2441,2442,2443,2444,2445,2446,2447,2448,2449,2450,2451,2452,2453,2454,2455,2456,2457,2458,2459,2460,2461,2462,2463,2464,2465,2466,2467,2468,2469,2470,2471,2472,2473,2474,2475,24

,index,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,SRW07,SRW08,SRW09,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122,...,0.00000,8.683555,0.000000,57.302060,943,45,124.0,154.0,8.791666666666666,4.638889
1,1,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120,...,0.00000,8.623174,0.000000,56.028098,834,42,118.0,146.0,7.930555555555555,4.513889
2,2,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84,...,0.00000,7.359468,0.000000,45.147991,458,15,64.0,67.0,4.972222222222222,3.666667
3,3,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54,...,0.00000,6.708084,0.000000,36.195345,88,10,38.0,39.0,2.861111111111111,2.166667
4,4,7.295906,-4.212500,7.295906,3.689676,0.542862,132.203,116.075,132.115030,56,...,0.00000,6.947937,0.000000,36.755147,96,8,36.0,36.0,5.0625,2.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3740,3742,11.101308,-3.927153,11.101308,0.000000,0.320094,216.121,207.049,216.024597,76,...,0.00000,7.822044,0.000000,45.446547,1300000226,21,64.0,75.0,divide by zero encountered in power (mZagreb1),2.888889
3741,3743,9.219634,-5.663419,9.219634,3.660438,0.167583,500.494,464.206,500.210506,200,...,0.00000,8.849227,0.000000,70.524683,3519,63,176.0,212.0,14.055555555555555,7.722222
3742,3744,7.461968,-2.825116,7.461968,0.150451,0.576897,154.238,144.158,154.056469,54,...,0.00000,7.311218,0.000000,39.310842,116,12,46.0,51.0,4.333333333333334,2.361111
3743,3745,6.354167,-2.812500,6.354167,2.812500,0.326161,64.084,56.020,64.052429,28,...,0.00000,1.609438,0.000000,12.047190,400000002,0,4.0,2.0,4.0,2.000000


In [19]:
or_f.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3745 entries, 0 to 3744
Columns: 3223 entries, index to mZagreb2
dtypes: bool(2), float64(2144), int64(364), object(713)
memory usage: 92.0+ MB


In [20]:
or_f1=or_f.drop(columns=['smiles','IsomericSMILES','name','Name','index','Unnamed: 0.1','Unnamed: 0'])

In [21]:
dta=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/update_11jan/COMBINE_VIETORIS_RIPS_PERSISTENCE_ENTROPY_OUT-edit.csv')
dta
label=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/update_11jan/combine_label.csv')
label_f=label.drop(columns=['name','Unnamed: 0'])
label_f
CID=label_f[['CID']]
f_dta=pd.concat([CID,dta],axis=1)
f_dta

,CID,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,3.904445,2.729808,0.693058
1,101602.0,3.947052,2.529469,0.485423
2,22235151.0,3.424074,1.140838,0.000000
3,68074566.0,3.067019,1.568200,0.000000
4,31361.0,3.141797,1.336667,0.000000
...,...,...,...,...
3781,24847856.0,4.197261,2.376034,0.282798
3782,44134940.0,2.905733,1.031380,0.000000
3783,55251157.0,2.897674,0.581067,0.000000
3784,91001317.0,2.330120,0.000000,0.000000


In [22]:
f_dta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3786 entries, 0 to 3785
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CID                        3786 non-null   float64
 1   PERSISTENCE_ENTROPY_DIM_0  3786 non-null   float64
 2   PERSISTENCE_ENTROPY_DIM_1  3786 non-null   float64
 3   PERSISTENCE_ENTROPY_DIM_2  3786 non-null   float64
dtypes: float64(4)
memory usage: 118.4 KB


In [ ]:
#f_dta['CID']=f_dta['CID'].astype(object)

In [ ]:
#or_f1_CID=or_f1['CID']
#f_dta_CID=f_dta['CID']

In [23]:
common_la = pd.merge(or_f1, f_dta, left_on='CID', right_on='CID', how='right')

# Rename the columns in the new DataFrame (optional)
common_la.rename(columns={'CID': 'CommonCID'}, inplace=True)
#common_la.rename(columns={'IsomericSMILES_x': 'IsomericSMILES'}, inplace=True)
# Display the common elements in a new DataFrame
#print(common_CID)
#common_la.to_csv('common_la.csv',index=False)
common_la

,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,NumRadicalElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,0.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,0.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,0.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,0.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3781,9.219634,-5.663419,9.219634,3.660438,0.167583,500.494,464.206,500.210506,200.0,0.0,...,70.524683,3519.0,63.0,176.0,212.0,14.055555555555555,7.722222,4.197261,2.376034,0.282798
3782,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.905733,1.031380,0.000000
3783,7.461968,-2.825116,7.461968,0.150451,0.576897,154.238,144.158,154.056469,54.0,0.0,...,39.310842,116.0,12.0,46.0,51.0,4.333333333333334,2.361111,2.897674,0.581067,0.000000
3784,6.354167,-2.812500,6.354167,2.812500,0.326161,64.084,56.020,64.052429,28.0,0.0,...,12.047190,400000002.0,0.0,4.0,2.0,4.0,2.000000,2.330120,0.000000,0.000000


In [24]:
com_f1=f_com_all.drop(columns=['name','IsomericSMILES'])
com_f1

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,177.0,9.569444,-2.722222,9.569444,1.506944,0.355008,44.053,40.021,44.026215,18,...,6.289459,4,0,6.0,4.0,2.250000,1.000000,1.777983,0.000000,0.000000
1,83036.0,7.970706,-4.726701,7.970706,4.177266,0.561161,202.338,176.130,202.193280,86,...,5.054832,397,12,56.0,55.0,6.833333,3.333333,3.606813,1.193387,0.000000
2,7765.0,7.456250,-3.900000,7.456250,3.527176,0.519424,118.176,104.064,118.099380,50,...,5.368154,75,6,28.0,27.0,4.111111,2.166667,2.999655,0.000000,0.000000
3,5367767.0,7.987270,-4.530029,7.987270,1.951380,0.317337,226.360,200.152,226.193280,94,...,5.385554,631,14,60.0,59.0,6.111111,4.166667,3.681115,1.027111,0.000000
4,66929.0,7.664583,-4.225000,7.664583,3.723747,0.534645,146.230,128.086,146.130680,62,...,5.218953,149,8,36.0,35.0,4.611111,2.666667,3.236267,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48,...,6.162052,65,8,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
1507,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86,...,7.402974,446,22,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
1508,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92,...,7.306372,702,23,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
1509,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30,...,5.076679,15,0,20.0,20.0,1.250000,1.250000,2.553369,0.244144,0.000000


In [25]:
com_f1 = com_f1.loc[:,~com_f1.columns.duplicated()].copy()

In [26]:
com_f1

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,177.0,9.569444,-2.722222,9.569444,1.506944,0.355008,44.053,40.021,44.026215,18,...,17.310771,4,0,6.0,4.0,2.250000,1.000000,1.777983,0.000000,0.000000
1,83036.0,7.970706,-4.726701,7.970706,4.177266,0.561161,202.338,176.130,202.193280,86,...,43.056109,397,12,56.0,55.0,6.833333,3.333333,3.606813,1.193387,0.000000
2,7765.0,7.456250,-3.900000,7.456250,3.527176,0.519424,118.176,104.064,118.099380,50,...,33.420942,75,6,28.0,27.0,4.111111,2.166667,2.999655,0.000000,0.000000
3,5367767.0,7.987270,-4.530029,7.987270,1.951380,0.317337,226.360,200.152,226.193280,94,...,45.012208,631,14,60.0,59.0,6.111111,4.166667,3.681115,1.027111,0.000000
4,66929.0,7.664583,-4.225000,7.664583,3.723747,0.534645,146.230,128.086,146.130680,62,...,36.593370,149,8,36.0,35.0,4.611111,2.666667,3.236267,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48,...,35.071670,65,8,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
1507,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86,...,47.511782,446,22,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
1508,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92,...,50.261825,702,23,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
1509,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30,...,41.004802,15,0,20.0,20.0,1.250000,1.250000,2.553369,0.244144,0.000000


In [27]:
common_la.rename(columns={'CommonCID': 'CID'}, inplace=True)
common_la.insert(0, 'CID', common_la.pop('CID'))
common_la

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3781,24847856.0,9.219634,-5.663419,9.219634,3.660438,0.167583,500.494,464.206,500.210506,200.0,...,70.524683,3519.0,63.0,176.0,212.0,14.055555555555555,7.722222,4.197261,2.376034,0.282798
3782,44134940.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.905733,1.031380,0.000000
3783,55251157.0,7.461968,-2.825116,7.461968,0.150451,0.576897,154.238,144.158,154.056469,54.0,...,39.310842,116.0,12.0,46.0,51.0,4.333333333333334,2.361111,2.897674,0.581067,0.000000
3784,91001317.0,6.354167,-2.812500,6.354167,2.812500,0.326161,64.084,56.020,64.052429,28.0,...,12.047190,400000002.0,0.0,4.0,2.0,4.0,2.000000,2.330120,0.000000,0.000000


In [28]:
com_f1.columns.difference(common_la.columns)

Index([], dtype='object')

In [29]:
all_m_feature=pd.concat([common_la, com_f1])
all_m_feature

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48.0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
1507,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86.0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
1508,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92.0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
1509,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30.0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


In [30]:
all_m_feature

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48.0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
1507,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86.0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
1508,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92.0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
1509,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30.0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


### All features data save as csv

In [ ]:
all_m_feature.to_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/all_data_feature_c.csv',index=False)

In [34]:
#label_final=pd.read_excel('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/same_all_a.xlsx')
#label_final=label_final.drop(label_final.columns[[0]],axis=1)
label_final=pd.read_excel('same_all_b.xlsx')
label_final

,CID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,10569,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,101602,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,22235151,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,68074566,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,31361,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5252,6781,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5253,8615,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
5254,31268,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [35]:
label_all=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/label_all.csv')
label_all

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
label_all.loc[:, (label_all != 0).any(axis=0)]  # https://stackoverflow.com/questions/21164910/how-do-i-delete-a-column-that-contains-only-zeros-in-pandas

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
label_all.to_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/label_all_a.csv')

### FUll dataset

In [43]:
all_m_feature=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/all_data_feature_c.csv')
all_m_feature

<ipython-input-43-fcbe9ef9e51d>:1: DtypeWarning: Columns (2085,2086,2087,2088,2089,2090,2091,2092,2093,2094,2095,2096,2156,2157,2158,2159,2160,2165,2166,2167,2168,2169,2174,2175,2176,2177,2178,2183,2184,2185,2186,2187,2192,2193,2194,2195,2196,2201,2202,2203,2204,2205,2264,2265,2266,2267,2268,2273,2274,2275,2276,2277,2282,2283,2284,2285,2286,2291,2292,2293,2294,2295,2300,2301,2302,2303,2304,2309,2310,2311,2312,2313,2317,2318,2319,2320,2321,2325,2326,2327,2328,2329,2333,2334,2335,2336,2337,2341,2342,2343,2344,2345,2349,2350,2351,2352,2353,2357,2358,2359,2360,2361,2365,2366,2367,2368,2369,2373,2374,2375,2376,2377,2381,2382,2383,2384,2385,2389,2390,2391,2392,2393,2397,2398,2399,2400,2401,2405,2406,2407,2408,2409,2413,2414,2415,2416,2417,2418,2419,2420,2421,2422,2423,2424,2425,2426,2427,2428,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2440,2441,2442,2443,2444,2445,2446,2447,2448,2449,2450,2451,2452,2453,2454,2455,2456,2457,2458,2459,2460,2461,2462,2463,2464,2465,2466,2467,2468,24

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5292,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48.0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5293,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86.0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5294,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92.0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5295,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30.0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


In [42]:
f_full=all_m_feature.drop_duplicates(subset=['CID'],keep='first')
f_full

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5292,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48.0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5293,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86.0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5294,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92.0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5295,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30.0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


In [44]:
label_final_full=label_all.drop_duplicates(subset=['CID'],keep='first')
label_final_full

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:

label_final_full=label_final.drop_duplicates(subset=['CID'],keep='first')
label_final_full

,CID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,10569,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,101602,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,22235151,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,68074566,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,31361,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5252,6781,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5253,8615,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
5254,31268,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
common_full = pd.merge(all_m_feature, label_final_full, on='CID',how='inner')

# Rename the columns in the new DataFrame (optional)
common_full.rename(columns={'CID': 'CommonCID'}, inplace=True)
#common_la.rename(columns={'IsomericSMILES_x': 'IsomericSMILES'}, inplace=True)
# Display the common elements in a new DataFrame
#print(common_CID)
#common_la.to_csv('common_la.csv',index=False)
common_full

,CommonCID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,0,0,0,0,0,0,0,0,0,0
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,0,0,0,0,0,0,0,0,0,0
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,0,0,0,0,0,0,0,0,0,0
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48.0,...,0,0,0,0,0,0,0,0,0,0
5279,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86.0,...,0,0,0,0,0,0,0,0,0,0
5280,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92.0,...,0,0,0,0,0,1,0,0,0,0
5281,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30.0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
common_full = label_final_full.merge(all_m_feature, on='CID',how='inner')

common_full.rename(columns={'CID': 'CommonCID'}, inplace=True)

common_full

,CommonCID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602,0,0,0,0,0,0,0,0,0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151,0,0,0,0,0,0,0,0,0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566,0,0,0,0,0,0,0,0,0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361,0,0,0,0,0,0,0,0,0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,6287,0,0,0,0,0,0,0,0,0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5279,6781,0,0,0,0,0,0,0,0,0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5280,8615,0,0,0,0,0,0,0,0,0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5281,31268,0,0,0,0,0,0,0,0,0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


## Drop dulicated CommonCID when combine

In [48]:
# Find rows with the same values in 'Column1' and 'Column2'
duplicates = common_full[common_full.duplicated(subset=['CommonCID'],keep=False)]  # if use keep=False : Returns ALL duplicated rows
duplicates

,CommonCID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
51,22877,0,0,0,0,0,0,0,0,0,...,2.000000,100000000.0,0.0,0.0,0.0,divide by zero encountered in power (mZagreb1),0.000000,2.969113,1.532242,0.0
52,22877,0,0,0,0,0,0,0,0,0,...,2.000000,100000000.0,0.0,0.0,0.0,divide by zero encountered in power (mZagreb1),0.000000,2.967706,1.390122,0.0
115,235,0,0,0,0,0,0,0,0,0,...,44.415409,171.0,19.0,68.0,82.0,4.736111111111111,2.500000,2.909647,0.559289,0.0
116,235,0,0,0,0,0,0,0,0,0,...,44.415409,171.0,19.0,68.0,82.0,4.736111111111111,2.500000,2.916352,1.045731,0.0
123,4650,0,0,0,0,0,1,0,0,0,...,30.941317,27.0,3.0,24.0,24.0,1.5,1.500000,2.395281,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5253,5362833,0,0,0,0,0,0,0,0,0,...,35.643477,165.0,7.0,34.0,32.0,4.0,2.750000,3.164434,0.000000,0.0
5264,5365069,0,0,0,0,0,0,0,0,0,...,41.681510,318.0,13.0,50.0,51.0,5.972222,3.361111,3.431464,0.000000,0.0
5265,5365069,0,0,0,0,0,0,0,0,0,...,41.681510,318.0,13.0,50.0,51.0,5.972222,3.361111,3.433145,0.684065,0.0
5270,5367719,0,0,0,0,0,0,0,0,0,...,61.134444,482.0,19.0,72.0,80.0,6.444444,3.888889,3.513325,0.665065,0.0


In [49]:
data_full=common_full.drop_duplicates(subset=['CommonCID'],keep='first')
data_full = data_full.iloc[1:]
data_full

,CommonCID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
1,101602,0,0,0,0,0,0,0,0,0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151,0,0,0,0,0,0,0,0,0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566,0,0,0,0,0,0,0,0,0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361,0,0,0,0,0,0,0,0,0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
5,6342,0,0,1,0,0,0,0,0,0,...,36.755147,96.0,8.0,36.0,36.0,5.0625,2.250000,1.585853,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,6287,0,0,0,0,0,0,0,0,0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5279,6781,0,0,0,0,0,0,0,0,0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5280,8615,0,0,0,0,0,0,0,0,0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5281,31268,0,0,0,0,0,0,0,0,0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


In [50]:
data_full.iloc[:, 1:201]

,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,coumarin,...,BCUT2D_MRHI,BCUT2D_MRLOW,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v
1,0,0,0,0,0,0,0,0,0,0,...,-9.990778e-17,-1.140125,3.157731,1603.705113,42.093858,40.816497,10.816497,22.290782,20.816497,5.908248
2,0,0,0,0,0,0,0,0,0,0,...,-2.665374e-16,-1.175233,3.224658,1622.177129,44.861807,44.408248,10.408248,23.337369,22.612372,5.704124
3,0,0,0,0,0,0,0,0,0,0,...,-9.645407e-17,-1.064829,3.585884,860.122253,26.585422,25.224745,7.224745,14.296747,12.474745,3.474745
4,0,0,0,0,0,0,0,0,0,0,...,-2.381637e-16,-1.127651,3.721251,446.007679,18.914214,18.316497,4.316497,9.664214,9.066497,2.066497
5,0,0,1,0,0,0,0,0,0,0,...,-2.163137e-16,-1.111385,6.520178,418.835803,20.914214,20.316497,4.316497,10.414214,9.816497,1.816497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,0,0,0,0,0,0,0,0,0,0,...,-1.896536e-16,-1.110326,6.347027,333.975158,15.861807,14.763710,3.763710,8.174756,6.934531,1.631855
5279,0,0,0,0,0,0,0,0,0,0,...,8.511220e-02,-1.053404,3.655531,798.389315,24.033016,21.632993,7.632993,13.154372,10.724745,3.724745
5280,0,0,0,0,0,0,0,0,0,0,...,1.871828e+00,-0.928420,2.673860,1118.332356,25.790011,23.763710,8.763710,14.895347,11.980406,4.585979
5281,0,0,0,0,0,0,0,0,0,0,...,-3.110023e-01,-1.015729,3.384671,231.537569,11.577350,11.447214,2.447214,5.904701,5.644427,1.197214


In [51]:
data_full_a = data_full.loc[(data_full.iloc[:, :165] != 0).any(axis=1)]
data_full_a

,CommonCID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
1,101602,0,0,0,0,0,0,0,0,0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151,0,0,0,0,0,0,0,0,0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566,0,0,0,0,0,0,0,0,0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361,0,0,0,0,0,0,0,0,0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
5,6342,0,0,1,0,0,0,0,0,0,...,36.755147,96.0,8.0,36.0,36.0,5.0625,2.250000,1.585853,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,6287,0,0,0,0,0,0,0,0,0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5279,6781,0,0,0,0,0,0,0,0,0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5280,8615,0,0,0,0,0,0,0,0,0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5281,31268,0,0,0,0,0,0,0,0,0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


## Final label and Final features

In [53]:
label_2501=data_full.iloc[:,:165]
label_2501

,CommonCID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,gardenia,butterscotch,peanut,sage,quince,blossom,raspberry,fennel,iris,corn
1,101602,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,22235151,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,68074566,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,31361,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,6342,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,6287,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5279,6781,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5280,8615,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
5281,31268,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [54]:
label_2501.loc[:, (label_2501 != 0).any(axis=0)]

,CommonCID,alcohol,aldehidic,almond,anise,beef,butter,cheese,citrus,coco,...,gardenia,butterscotch,peanut,sage,quince,blossom,raspberry,fennel,iris,corn
1,101602,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,22235151,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,68074566,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,31361,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,6342,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,6287,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5279,6781,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5280,8615,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
5281,31268,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
all_data=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/all_data.csv')
all_data

,name,IsomericSMILES,CID
0,abietic acid,CC(C)C1=CC2=CC[C@@H]3[C@@]([C@H]2CC1)(CCC[C@@]...,10569.0
1,dihydroabietyl alcohol,CC(C)C1CCC2C(=C1)CCC3C2(CCCC3(C)CO)C,101602.0
2,acetaldehyde,CC=O,177.0
3,benzyl methoxyethyl acetal,CC(OCCOC)OCC1=CC=CC=C1,22235151.0
4,13002-09-0,CC(C)CCOC(C)OCCC(C)C,83036.0
...,...,...,...
7213,calcium saccharin,C1=CC=C2C(=C1)C(=O)[N-]S2(=O)=O.C1=CC=C2C(=C1)...,44134940.0
7214,erythorbic acid,C([C@H]([C@@H]1C(=C(C(=O)O1)O)O)O)O,54675810.0
7215,67952-65-2,CC1=CN=C(C(=N1)C)SC,55251157.0
7216,"pyroligneous acids, reaction products with et ...",CC.OO,91001317.0


In [57]:
Feature_2501=data_full.iloc[:,165:]
Feature_2501

,mango,lavender,beer,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
1,0,0,0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,0,0,0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,0,0,0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,0,0,0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
5,0,0,0,7.295906,-4.212500,7.295906,3.689676,0.542862,132.203,116.075,...,36.755147,96.0,8.0,36.0,36.0,5.0625,2.250000,1.585853,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,0,0,0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5279,0,0,0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5280,0,0,0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5281,0,0,0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


In [ ]:
Feature_2501.to_csv('feature_2501.csv',index=False)
label_2501.to_csv('label_2501.csv',index=False)